# Construct End-User HDF5 Files

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import uproot
import glob
import awkward as ak
import itertools
import yaml
import os
import sys
from tqdm import tqdm
from pathlib import Path
import atlasify as atl
from typing import List
import h5py
atl.ATLAS = "ColliderML"

sys.path.append("../")
sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/OtherLibraries/pyedm4hep")
from pyedm4hep import EDM4hepEvent
from utils import load_root_file

## Roadmap

1. Load in edm4hep file
4. Load in measurements root
5. Load in trackstates ambi roo
8. Digihit object needs:
    - ...

# Loading

In [2]:
base_dir = "/global/cfs/cdirs/m4958/data/ColliderML/outputs/low_pileup_pilot/gg2ttbar/v1/runs/0"
event_num = 0

## 1. Load in edm4hep file

In [3]:
event = EDM4hepEvent(os.path.join(base_dir, "edm4hep.root"), event_index=event_num)

Loading event 0 from /global/cfs/cdirs/m4958/data/ColliderML/outputs/low_pileup_pilot/gg2ttbar/v1/runs/0/edm4hep.root...
  Loaded 406505 particles.
  Loaded 22065 tracker hits.
  Loaded 178292 calo hits and 761509 contributions.


In [4]:
tracker_df = event.get_tracker_hits_df()
tracker_df.columns

# Cast x,y,z to float32
tracker_df[["x", "y", "z"]] = tracker_df[["x", "y", "z"]].astype(np.float32)

# Sort by x,y,z
tracker_df = tracker_df.sort_values(by=["x", "y", "z"])

# DigiHits Objects

In [5]:
digihits_path = "/global/cfs/cdirs/m4958/data/ColliderML/outputs/low_pileup_pilot/gg2ttbar/v1/runs/0/measurements.root"
simhits_path = "/global/cfs/cdirs/m4958/data/ColliderML/outputs/low_pileup_pilot/gg2ttbar/v1/runs/0/simhits.root"

In [6]:
# uproot load digihits file

digihits_df = load_root_file(digihits_path)
simhits_df = load_root_file(simhits_path)

# Cast true_x,true_y,true_z to float32
digihits_df[["true_x", "true_y", "true_z"]] = digihits_df[["true_x", "true_y", "true_z"]].astype(np.float32)

# Sort by true_x,true_y,true_z
digihits_df = digihits_df.sort_values(by=["true_x", "true_y", "true_z"])

In [7]:
digihits_df = digihits_df[digihits_df.event_nr == 0]
simhits_df = simhits_df[simhits_df.event_id == 0]

In [8]:
digihits_df[["true_x", "true_y", "true_z"]]

,true_x,true_y,true_z
entry,,,
16877,-1063.397705,62.497215,-3009.593262
20871,-1061.386841,102.830124,1604.617554
21991,-1059.093262,-60.923317,3025.500000
21996,-1057.518799,-61.942005,3020.500000
17339,-1056.741943,-125.997696,-2275.500000
...,...,...,...
17494,1060.122437,2.137749,-1909.500000
16832,1060.265381,63.115620,-3004.571289
21151,1060.539185,-82.798782,1909.500000


First, does digihits and simhits match up?

In [9]:
tracker_df[["x", "y", "z"]]

,x,y,z
20243,-1063.397705,62.497215,-3009.593262
20836,-1061.386841,102.830124,1604.617554
21250,-1059.093262,-60.923317,3025.500000
21249,-1057.518799,-61.942005,3020.500000
19313,-1056.741943,-125.997696,-2275.500000
...,...,...,...
20017,1060.122437,2.137749,-1909.500000
20440,1060.265381,63.115620,-3004.571289
21112,1060.539185,-82.798782,1909.500000
20450,1063.784058,-84.316940,-2620.500000


In [10]:
# Merge digihits_df with tracker_df on left=true_x,true_y,true_z right=x,y,z

merged = pd.merge(digihits_df, tracker_df, left_on=["true_x", "true_y", "true_z"], right_on=["x", "y", "z"], how="left")

In [11]:
merged

,event_nr,volume_id,layer_id,surface_id,rec_loc0,rec_loc1,rec_time,var_loc0,var_loc1,var_time,...,pz,EDep,particle_id,detector,r,R,phi,theta,eta,pt
0,0,28,2,132,-63.941242,NaN,NaN,0.005184,NaN,NaN,...,0.022248,0.000017,284097,LongStripEndcapReadout,1065.232620,3192.549500,3.082889,2.801407,-1.761702,0.043370
1,0,30,4,129,-38.022579,NaN,NaN,0.005184,NaN,NaN,...,-0.000013,0.000005,97309,LongStripEndcapReadout,1066.356413,1926.632651,3.045011,0.586539,1.197404,0.000033
2,0,30,12,131,59.553139,NaN,NaN,0.005184,NaN,NaN,...,1.426032,0.000072,3358,LongStripEndcapReadout,1060.844046,3206.094250,-3.084132,0.337240,1.770567,0.534349
3,0,30,12,132,63.235840,NaN,NaN,0.005184,NaN,NaN,...,1.426972,0.000066,3358,LongStripEndcapReadout,1059.331326,3200.875366,-3.083087,0.337311,1.770353,0.535118
4,0,28,6,133,14.583749,NaN,NaN,0.005184,NaN,NaN,...,0.261832,0.000495,66624,LongStripEndcapReadout,1064.226902,2512.066709,-3.022921,2.704126,-1.503774,0.352577
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22060,0,28,8,98,0.631801,NaN,NaN,0.005184,NaN,NaN,...,-2.188817,0.000075,2761,LongStripEndcapReadout,1060.124570,2184.045410,0.002017,2.634778,-1.351025,1.256339
22061,0,28,2,97,64.598694,NaN,NaN,0.005184,NaN,NaN,...,0.000025,0.000027,404443,LongStripEndcapReadout,1062.142284,3186.784308,0.059458,2.801795,-1.762866,0.000080
22062,0,30,6,182,-54.852924,NaN,NaN,0.005184,NaN,NaN,...,0.075022,0.000142,152520,LongStripEndcapReadout,1063.766383,2185.815447,-0.077914,0.508271,1.348028,0.077788
22063,0,28,4,182,-56.810757,NaN,NaN,0.005184,NaN,NaN,...,-0.068347,0.000089,404550,LongStripEndcapReadout,1067.120364,2829.446257,-0.079096,2.754878,-1.630642,0.063712


In [41]:
digihits_df = digihits_df.sort_values(by=["true_x", "true_y", "true_z"])
tracker_df = tracker_df.sort_values(by=["x", "y", "z"])

In [44]:
digihits_df[["true_x", "true_y", "true_z"]]

,true_x,true_y,true_z
entry,,,
16877,-1063.397705,62.497215,-3009.593262
20871,-1061.386841,102.830124,1604.617554
21991,-1059.093262,-60.923317,3025.500000
21996,-1057.518799,-61.942005,3020.500000
17339,-1056.741943,-125.997696,-2275.500000
...,...,...,...
17494,1060.122437,2.137749,-1909.500000
16832,1060.265381,63.115620,-3004.571289
21151,1060.539185,-82.798782,1909.500000


In [45]:
tracker_df[["x", "y", "z"]]

,x,y,z
20243,-1063.397684,62.497214,-3009.593291
20836,-1061.386813,102.830122,1604.617516
21250,-1059.093216,-60.923316,3025.500000
21249,-1057.518816,-61.942006,3020.500000
19313,-1056.741917,-125.997695,-2275.500000
...,...,...,...
20017,1060.122415,2.137749,-1909.500000
20440,1060.265368,63.115618,-3004.571183
21112,1060.539146,-82.798780,1909.500000
20450,1063.784060,-84.316937,-2620.500000


## Local to Global Conversion

In [ ]:
import uproot
import numpy as np

import acts
import acts.examples as ax



ModuleNotFoundError: No module named 'acts'

In [ ]:
# 1) Load the same geometry
geo_dir = ax.odd.getOpenDataDetectorDirectory()
det = ax.odd.getOpenDataDetector(odd_dir=geo_dir, materialDecorator=acts.IMaterialDecorator.fromFile(geo_dir / "data/odd-material-maps.root"))
tg = det.trackingGeometry()

# 2) Build converter with a minimal config: only surfaceByIdentifier is required
cfg = ax.DigitizationAlgorithm.Config()
cfg.surfaceByIdentifier = tg.geoIdSurfaceMap()
conv = ax.DigitizationCoordinatesConverter(cfg)

# 3) Read measurements.root
with uproot.open("/path/to/measurements.root") as f:
    t = f["measurements"]
    vol = t["volume_id"].array(library="np")
    lay = t["layer_id"].array(library="np")
    sen = t["surface_id"].array(library="np")
    ext = t["extra_id"].array(library="np")
    loc0 = t["rec_loc0"].array(library="np")
    loc1 = t["rec_loc1"].array(library="np")

# 4) Rebuild geometry id and convert
def make_gid(v, l, s, e):
    gid = acts.GeometryIdentifier()
    gid = gid.withVolume(int(v)).withLayer(int(l)).withSensitive(int(s)).withExtra(int(e))
    return gid.value()

gids = np.array([make_gid(v, l, s, e) for v, l, s, e in zip(vol, lay, sen, ext)], dtype=np.uint64)

xyz = np.array([conv.localToGlobal(int(g), float(x), float(y)) for g, x, y in zip(gids, loc0, loc1)], dtype=float)
x, y, z = xyz[:,0], xyz[:,1], xyz[:,2]